# Deepfake Detection — CUI Lahore
FA23-BCS-107 | MUHAMMAD FAHAD HUSSAIN RANA     
FA23-BCS-099 | MUHAMMAD AHSAN SHAIKH     
FA23-BCS-091 | MUHAMMAD ABDULLAH 

Trains all three models on a T4/A100 GPU utilizing dynamic hardware scaling.

**Before running:** Runtime → Change runtime type → T4 GPU

In [ ]:
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('NOT FOUND — change runtime to GPU')

## 1. Clone repo & install dependencies

In [ ]:
!git clone https://github.com/MuhammadFad/DeepfakeDetection.git
%cd DeepfakeDetection
!pip install timm opencv-python plotly scikit-learn tqdm requests -q
!pip install grad-cam --no-deps -q

## 2. Download Kaggle dataset
Get your API key: kaggle.com → Settings → API → **Create New Token** → upload `kaggle.json` when prompted.

In [ ]:
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -q
!unzip -q 140k-real-and-fake-faces.zip
!ls real_vs_fake/real-vs-fake/

DATASET_ROOT = 'real_vs_fake/real-vs-fake'

## 3. Populate image folders (5k train / 1k val / 1k test per class)

In [ ]:
from PIL import Image
from pathlib import Path

SRC    = Path(DATASET_ROOT)
DST    = Path('images')
COUNTS = {'train': 5000, 'valid': 1000, 'test': 1000}
MAP    = {'valid': 'val'}
TARGET_SIZE = 299  # pre-resize to largest model input; ViT resizes 299→224 during cache fill

for src_split, n in COUNTS.items():
    dst_split = MAP.get(src_split, src_split)
    for cls in ('real', 'fake'):
        src_dir = SRC / src_split / cls
        dst_dir = DST / dst_split / cls
        dst_dir.mkdir(parents=True, exist_ok=True)
        files_list = sorted(src_dir.iterdir())[:n]
        copied = 0
        for f in files_list:
            dst_path = dst_dir / f.name
            if dst_path.exists():
                continue
            img = Image.open(f).convert('RGB')
            img = img.resize((TARGET_SIZE, TARGET_SIZE), Image.BILINEAR)
            img.save(dst_path, 'JPEG', quality=90)
            copied += 1
        print(f'  {dst_split}/{cls}: {len(files_list)} images ({copied} new)')

## 4. Set Training Flags

In [ ]:
import re

config_text = open('src/config.py').read()
current_val = int(re.search(r'BATCH_SIZE_TRAIN\s*=\s*(\d+)', config_text).group(1))
if current_val != 64:
    config_text = re.sub(r'(BATCH_SIZE_TRAIN\s*=\s*)\d+', r'\g<1>64', config_text)
    open('src/config.py', 'w').write(config_text)
val = re.search(r'BATCH_SIZE_TRAIN\s*=\s*(\d+)', open('src/config.py').read()).group(1)
print(f'Batch size: {val}')

RETRAIN_XCEPTION     = True
RETRAIN_VIT          = True
RETRAIN_EFFICIENTNET = True

## 5. Train models

In [ ]:
flag = '--force' if (RETRAIN_XCEPTION or RETRAIN_VIT or RETRAIN_EFFICIENTNET) else '--skip-existing'
!python scripts/train.py --model xception --auto-scale --cache {flag}
!python scripts/train.py --model vit_small_patch16_224 --auto-scale --cache {flag}
!python scripts/train.py --model efficientnet_b4 --auto-scale --cache {flag}

## 6. Download checkpoints
Place the downloaded `.pth` files in `checkpoints/` and push to GitHub with Git LFS.

In [ ]:
from google.colab import files
import os

for name in ['xception_best.pth', 'vit_small_patch16_224_best.pth', 'efficientnet_b4_best.pth']:
    path = f'checkpoints/{name}'
    if os.path.exists(path):
        print(f'Downloading {name}...')
        files.download(path)
    else:
        print(f'MISSING: {path} — training may have failed')

print('\nAlso downloading training log...')
files.download('output/training_log.txt')